<a href="https://colab.research.google.com/github/mahdisahad/WOA-Feature-Selection/blob/main/notebooks/03_Baseline_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="text-align: right">

# نوت‌بوک ۳: مدل‌های پایه (Baseline Models)

قبل از اینکه ببینیم الگوریتم نهنگ چه کمکی می‌کند، باید بدانیم **بدون انتخاب ویژگی** و با **همه‌ی ویژگی‌ها** مدل‌های معمول چه نتیجه‌ای می‌دهند. این نتیجه معیار مقایسه (baseline) برای نوت‌بوک‌های بعدی است.

مراحل: بارگذاری و تقسیم داده، حذف ویژگی‌های تکراری و ثابت (فقط با train)، تعریف سه مدل، ارزیابی با اعتبارسنجی متقابل (CV) روی train، ارزیابی روی test، و ذخیره‌ی نتایج.

دو نوع نتیجه داریم و نباید با هم اشتباه شوند:
- **نتیجه‌ی CV:** فقط روی داده‌ی train و برای مقایسه و انتخاب مدل.
- **نتیجه‌ی test:** روی داده‌ای که هیچ‌جا برای تصمیم‌گیری استفاده نشده و برای گزارش نهایی است.

</div>

<div dir="rtl" style="text-align: right">

### اتصال به Drive، بارگذاری داده و جدا کردن X و y
Drive وصل می‌شود، تنظیمات دیتاست از `dataset_config.py` وارد می‌شود و `load_dataset` دیتاست فعال را می‌خواند. `split_xy` آن را به `X` (ویژگی‌ها)، `y` (برچسب) و `groups` (شناسه‌ی افراد، فقط برای Parkinson) تقسیم می‌کند. ابعاد و توزیع کلاس‌ها چاپ می‌شود.

</div>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

from pathlib import Path

PROJECT_PATH = Path("/content/drive/MyDrive/WOA_Project")
import sys
sys.path.append(str(PROJECT_PATH))
from dataset_config import *   # <-- dataset-specific code lives only here

df = load_dataset(PROJECT_PATH)

X, y, groups = split_xy(df)

print("Dataset shape:", X.shape)
print("Target shape:", y.shape)
print("Class distribution:")
print(y.value_counts())


<div dir="rtl" style="text-align: right">

### تقسیم به train و test
`split_train_test` داده را ۸۰٪ / ۲۰٪ تقسیم می‌کند (`random_state=42` و تقسیم طبقه‌ای؛ برای Parkinson تقسیم بر اساس فرد). همه‌ی تصمیم‌ها بعد از این‌جا فقط با train گرفته می‌شود و test برای ارزیابی نهایی کنار می‌ماند.

</div>

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, groups_train = split_train_test(X, y, groups)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)


<div dir="rtl" style="text-align: right">

### حذف ویژگی‌های تکراری و ثابت (فقط روی train)
- ستون‌های تکراری (دقیقاً برابر با یک ستون قبلی) با `X_train.T.duplicated()` پیدا و از train و test حذف می‌شوند.
- `drop_near_constant` ستون‌هایی را که در train تقریباً ثابت‌اند (انحراف معیار ≈ ۰) حذف می‌کند. چنین ستون‌هایی اطلاعاتی ندارند و فقط فضای جست‌وجو را بزرگ می‌کنند.

تصمیم هر دو حذف فقط از روی train گرفته می‌شود و برچسب‌ها هیچ نقشی ندارند. ابعاد بعد از حذف چاپ می‌شود؛ همین ابعاد در نوت‌بوک‌های ۰۴ تا ۰۶ هم ثابت است.

</div>

In [ ]:
duplicate_columns = X_train.columns[
    X_train.T.duplicated()
].tolist()

print("Number of duplicate features:", len(duplicate_columns))

X_train = X_train.drop(columns=duplicate_columns)
X_test = X_test.drop(columns=duplicate_columns)
X_train, X_test = drop_near_constant(X_train, X_test)

print("Training shape after duplicate removal:", X_train.shape)
print("Testing shape after duplicate removal:", X_test.shape)


<div dir="rtl" style="text-align: right">

### تعریف سه مدل پایه
سه مدل داخل یک دیکشنری تعریف شده‌اند و هر کدام یک `Pipeline` شامل استانداردسازی و طبقه‌بند هستند:
- **Logistic Regression:** مدل خطی احتمالاتی؛ `max_iter=5000` تا بهینه‌سازی به همگرایی برسد.
- **Linear SVM:** ماشین بردار پشتیبان با مرز تصمیم خطی.
- **RBF SVM:** ماشین بردار پشتیبان با هسته‌ی غیرخطی RBF.

نکته‌ها: `StandardScaler` داخل `Pipeline` است، پس در هر فولد CV فقط روی بخش train همان فولد fit می‌شود و نشت داده رخ نمی‌دهد. `class_weight="balanced"` وزن هر کلاس را معکوس فراوانی‌اش می‌کند تا کلاس‌های کم‌نمونه نادیده گرفته نشوند.

</div>

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            class_weight="balanced",
            max_iter=5000,
            random_state=42
        ))
    ]),

    "Linear SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(
            kernel="linear",
            class_weight="balanced"
        ))
    ]),

    "RBF SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(
            kernel="rbf",
            class_weight="balanced"
        ))
    ])
}

print("Baseline models defined successfully.")

<div dir="rtl" style="text-align: right">

### ارزیابی با اعتبارسنجی متقابل ۵-تایی (روی train)
`cv = make_cv(groups_train)` ۵ فولد با `shuffle` و `random_state=42` می‌سازد (برای Parkinson فولدها بر اساس فرد ساخته می‌شوند تا یک فرد هم در آموزش و هم در اعتبارسنجی نباشد). `cross_validate` هر مدل را ۵ بار آموزش و روی فولد کنار گذاشته‌شده ارزیابی می‌کند و میانگین سه معیار گزارش می‌شود:
- **Accuracy:** درصد پیش‌بینی‌های درست.
- **Balanced Accuracy:** میانگین recall هر کلاس؛ در داده‌ی نامتوازن قابل‌اعتمادتر است.
- **F1 Macro:** میانگین F1 همه‌ی کلاس‌ها با وزن برابر؛ همان معیاری که بعداً در fitness الگوریتم نهنگ استفاده می‌شود.

</div>

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = make_cv(groups_train)

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1_macro": "f1_macro"
}

cv_results = {}

for name, model in models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )

    cv_results[name] = {
        "Accuracy": scores["test_accuracy"].mean(),
        "Balanced Accuracy": scores["test_balanced_accuracy"].mean(),
        "F1 Macro": scores["test_f1_macro"].mean()
    }

results_df = pd.DataFrame(cv_results).T

print(results_df.round(4))


<div dir="rtl" style="text-align: right">

### انتخاب بهترین مدل پایه
مدلی که بیشترین F1 Macro در CV را دارد با `idxmax()` انتخاب می‌شود. انتخاب فقط با نتیجه‌ی CV انجام می‌شود، نه test. (در صورت برابری، اولین مدل در جدول انتخاب می‌شود.)

</div>

In [ ]:
best_model_name = results_df["F1 Macro"].idxmax()

print("Best baseline model based on CV F1 Macro:")
print(best_model_name)

print("\nBest model results:")
print(results_df.loc[best_model_name])


<div dir="rtl" style="text-align: right">

### ارزیابی هر سه مدل روی test
هر مدل روی کل train آموزش داده می‌شود (`fit`) و روی test پیش‌بینی می‌کند (`predict`). برای هر مدل Accuracy، Balanced Accuracy، Precision، Recall و F1 (به‌صورت Macro) محاسبه می‌شود. مدل‌های آموزش‌دیده در `fitted_models` نگه داشته می‌شوند تا دوباره استفاده شوند. این اعداد نتیجه‌ی واقعی و بدون‌سوگیری هستند، چون test در انتخاب هیچ چیزی دخالت نداشته است.

</div>

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

test_results = {}

fitted_models = {}

for name, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    fitted_models[name] = model

    test_results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
        "Precision Macro": precision_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
        "Recall Macro": recall_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
        "F1 Macro": f1_score(
            y_test, y_pred, average="macro", zero_division=0
        )
    }

test_results_df = pd.DataFrame(test_results).T

print(test_results_df.round(4))

<div dir="rtl" style="text-align: right">

### گزارش دقیق بهترین مدل
- **Classification Report:** برای هر کلاس Precision (از پیش‌بینی‌های این کلاس چند درصد درست بود)، Recall (از نمونه‌های واقعی این کلاس چند درصد پیدا شد)، F1 و تعداد نمونه (support).
- **Confusion Matrix:** ردیف‌ها کلاس واقعی و ستون‌ها کلاس پیش‌بینی‌شده‌اند؛ قطر اصلی پیش‌بینی‌های درست و بقیه خطاها هستند و نشان می‌دهد کدام کلاس‌ها با هم اشتباه می‌شوند.

</div>

In [ ]:
best_model = fitted_models[best_model_name]

y_pred_best = best_model.predict(X_test)

print("Best model:", best_model_name)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))


<div dir="rtl" style="text-align: right">

### ذخیره‌ی نتایج
`get_results_path` پوشه‌ی `results/<نام دیتاست>/` را می‌سازد. جدول نتایج CV در `baseline_cv_results.csv` و نتایج test در `baseline_test_results.csv` ذخیره می‌شود تا برای مقایسه‌ی نهایی در نوت‌بوک‌های بعدی و گزارش در دسترس باشد.

</div>

In [ ]:
RESULTS_PATH = get_results_path(PROJECT_PATH)

results_df.to_csv(
    RESULTS_PATH / "baseline_cv_results.csv"
)

test_results_df.to_csv(
    RESULTS_PATH / "baseline_test_results.csv"
)

print("Baseline results saved successfully")
